In [0]:
%sql
SHOW TABLES IN healthcare_demo.silver;

In [0]:
from pyspark.sql import functions as F

silver = spark.table(
    "healthcare_demo.silver.patient_referral_details"
)

display(silver)

In [0]:
from pyspark.sql import functions as F

silver = spark.table("healthcare_demo.silver.patient_referral_details")

waiting_report = (
    silver
    .groupBy("specialty", "priority")
    .agg(
        F.countDistinct("referral_id").alias("total_referrals"),
        F.round(F.avg("waiting_days"), 1).alias("average_waiting_days"),
        F.max("waiting_days").alias("maximum_waiting_days")
    )
    .orderBy("specialty", "priority")
)

display(waiting_report)

In [0]:
# Unstructured referral document summary

clinical_report = (
    silver
    .groupBy(
        "clinical_symptoms",
        "document_priority",
        "document_specialty"
    )
    .agg(
        F.countDistinct("referral_id").alias("total_referrals"),
        F.round(F.avg("waiting_days"), 1).alias("average_waiting_days")
    )
    .orderBy(
        "document_specialty",
        "document_priority"
    )
)

display(clinical_report)

# Save Gold table

clinical_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_demo.gold.clinical_referral_summary")

print("Gold clinical referral summary saved successfully")

In [0]:
waiting_report.write.format("delta").mode("overwrite").saveAsTable(
    "healthcare_demo.gold.waiting_time_summary"
)

In [0]:
from pyspark.sql import functions as F

appointments = spark.table(
    "healthcare_demo.bronze.appointments"
)

capacity = spark.table(
    "healthcare_demo.bronze.capacity"
)

silver = spark.table(
    "healthcare_demo.silver.patient_referral_details"
)

In [0]:
referral_appointments = (
    appointments
    .join(
        silver.select("referral_id", "specialty"),
        on="referral_id",
        how="left"
    )
)

display(referral_appointments)

In [0]:
location_mapping = (
    capacity
    .select("location_id", "location_name")
    .distinct()
)

capacity_details = capacity.select(
    "location_id",
    "specialty",
    "available_slots"
)

capacity_enriched = (
    referral_appointments
    .join(
        location_mapping,
        on="location_id",
        how="left"
    )
    .join(
        capacity_details,
        on=["location_id", "specialty"],
        how="left"
    )
)

display(capacity_enriched)

In [0]:
capacity_report = (
    capacity_enriched
    .groupBy(
        "location_id",
        "location_name",
        "specialty"
    )
    .agg(
        F.countDistinct(
            F.when(
                F.lower(F.col("appointment_status")) != "cancelled",
                F.col("appointment_id")
            )
        ).alias("total_appointments"),
        F.max("available_slots").alias("available_slots")
    )
    .orderBy("location_id", "specialty")
)

display(capacity_report)

In [0]:
capacity_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_demo.gold.clinic_capacity_summary")

print("Gold clinic capacity summary saved successfully")

In [0]:
display(
    appointments.orderBy("appointment_date").limit(20)
)

In [0]:
display(capacity)

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)

audit_schema = StructType([
    StructField("check_name", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("failed_records", IntegerType(), False),
    StructField("result", StringType(), False),
    StructField("checked_at", TimestampType(), False)
])

empty_audit = spark.createDataFrame([], audit_schema)

empty_audit.write \
    .format("delta") \
    .mode("ignore") \
    .saveAsTable("healthcare_demo.gold.data_quality_audit")

print("Data quality audit table created")

In [0]:
recommendations = spark.table(
    "healthcare_demo.gold.appointment_recommendations"
)

print("Total recommendations:", recommendations.count())

display(recommendations)

In [0]:
recommendations = spark.table(
    "healthcare_demo.gold.appointment_recommendations"
)

display(
    recommendations
    .groupBy("specialty", "allocation_status")
    .count()
    .orderBy("specialty", "allocation_status")
)

In [0]:
display(
    recommendations
    .groupBy("referral_id")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
display(
    recommendations.filter(
        (F.col("allocation_status") == "Recommended") &
        (
            F.col("recommended_location_id").isNull() |
            F.col("recommended_location_name").isNull()
        )
    )
)